#### **Objectif : Prédire la probabilité qu'un client présente un risque de défaut de remboursement de son crédit**

### Chargement des bibliotheques

In [2]:
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
chemin = '../../data/raw/home-credit-default-risk'

### Chargement des donnees

In [3]:
df = pd.read_csv(f'{chemin}/application_train.csv')

#### Diagnostique de la base

In [4]:
print('Structure de la base')
print(df.info())
print('\nLes premiers observations')
print(df.head())

Structure de la base
<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), str(16)
memory usage: 286.2 MB
None

Les premiers observations
   SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR  \
0      100002       1         Cash loans           M            N   
1      100003       0         Cash loans           F            N   
2      100004       0    Revolving loans           M            Y   
3      100006       0         Cash loans           F            N   
4      100007       0         Cash loans           M            N   

  FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  \
0               Y             0          202500.0    406597.5      24700.5   
1               N             0          270000.0   1293502.5      35698.5   
2               Y             0           67500.0    135000.0       6750.0   
3               Y          

#### Analyse descriptive

In [18]:
## recuperer le nom des variable quantitatives
var_quanti = df.select_dtypes(include=np.number).columns.to_list()

## recuperer le nom des variables qualitatives
var_quali = df.select_dtypes(exclude=np.number).columns.to_list()

In [22]:
## verifier le nbre de modalite des variables quantitatives
m = 0
for var in var_quanti:
    if df[var].nunique() < 5:
        print(f"{var:<30} : {df[var].nunique():>10}")
        var_quali.append(var)
        var_quanti.remove(var)
        m += 1
print(m)
print(len(var_quanti))

TARGET                         :          2
FLAG_MOBIL                     :          2
FLAG_WORK_PHONE                :          2
FLAG_PHONE                     :          2
REGION_RATING_CLIENT           :          3
REG_REGION_NOT_LIVE_REGION     :          2
LIVE_REGION_NOT_WORK_REGION    :          2
REG_CITY_NOT_WORK_CITY         :          2
FLAG_DOCUMENT_2                :          2
FLAG_DOCUMENT_4                :          2
FLAG_DOCUMENT_6                :          2
FLAG_DOCUMENT_8                :          2
FLAG_DOCUMENT_10               :          2
FLAG_DOCUMENT_12               :          2
FLAG_DOCUMENT_14               :          2
FLAG_DOCUMENT_16               :          2
FLAG_DOCUMENT_18               :          2
FLAG_DOCUMENT_20               :          2
18
88


In [46]:
## verifier les modalites des variables qualitatives
for var in var_quali:
    print(f'{var:<40} {list(df[var].unique())}')

NAME_CONTRACT_TYPE                       ['Cash loans', 'Revolving loans']
CODE_GENDER                              ['M', 'F', 'XNA']
FLAG_OWN_CAR                             ['N', 'Y']
FLAG_OWN_REALTY                          ['Y', 'N']
NAME_TYPE_SUITE                          ['Unaccompanied', 'Family', 'Spouse, partner', 'Children', 'Other_A', nan, 'Other_B', 'Group of people']
NAME_INCOME_TYPE                         ['Working', 'State servant', 'Commercial associate', 'Pensioner', 'Unemployed', 'Student', 'Businessman', 'Maternity leave']
NAME_EDUCATION_TYPE                      ['Secondary / secondary special', 'Higher education', 'Incomplete higher', 'Lower secondary', 'Academic degree']
NAME_FAMILY_STATUS                       ['Single / not married', 'Married', 'Civil marriage', 'Widow', 'Separated', 'Unknown']
NAME_HOUSING_TYPE                        ['House / apartment', 'Rented apartment', 'With parents', 'Municipal apartment', 'Office apartment', 'Co-op apartment']
OCCUPAT

In [23]:
## statistique variable quantitative
df[var_quanti].describe().T

,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,307511.0,278180.518577,102790.175348,100002.0,189145.5,278202.0,367142.5,456255.0
CNT_CHILDREN,307511.0,0.417052,0.722121,0.0,0.0,0.0,1.0,19.0
AMT_INCOME_TOTAL,307511.0,168797.919297,237123.146279,25650.0,112500.0,147150.0,202500.0,117000000.0
AMT_CREDIT,307511.0,599025.999706,402490.776996,45000.0,270000.0,513531.0,808650.0,4050000.0
AMT_ANNUITY,307499.0,27108.573909,14493.737315,1615.5,16524.0,24903.0,34596.0,258025.5
...,...,...,...,...,...,...,...,...
AMT_REQ_CREDIT_BUREAU_DAY,265992.0,0.007000,0.110757,0.0,0.0,0.0,0.0,9.0
AMT_REQ_CREDIT_BUREAU_WEEK,265992.0,0.034362,0.204685,0.0,0.0,0.0,0.0,8.0
AMT_REQ_CREDIT_BUREAU_MON,265992.0,0.267395,0.916002,0.0,0.0,0.0,0.0,27.0
AMT_REQ_CREDIT_BUREAU_QRT,265992.0,0.265474,0.794056,0.0,0.0,0.0,0.0,261.0


#### Analyse de la variable cible

Index(['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER',
       'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL',
       'AMT_CREDIT', 'AMT_ANNUITY',
       ...
       'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20',
       'FLAG_DOCUMENT_21', 'AMT_REQ_CREDIT_BUREAU_HOUR',
       'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
       'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
       'AMT_REQ_CREDIT_BUREAU_YEAR'],
      dtype='str', length=122)


np.int64(24825)